Perfect 🚀
Let’s implement your **Order Management Agents project** with the requested scaffolding.
We’ll make **`server1` a real MCP client** that fetches weather from `https://wttr.in/{location}?format=3`.

---

# 📂 Project Structure

```
order_mgmt_agents/
│── main.py
│
├── config/
│   ├── __init__.py
│   ├── settings.py
│
├── tools/
│   ├── __init__.py
│   ├── weather_tools.py
│   ├── pollution_tools.py
│
├── mcp/
│   ├── __init__.py
│   ├── server1.py   (weather MCP client → wttr.in)
│   ├── server2.py   (dummy placeholder)
│   ├── server3.py   (dummy placeholder)
│   ├── server4.py   (dummy placeholder)
│
└── agents/
    ├── __init__.py
    ├── agent_factory.py
    ├── weather_agent.py
    ├── pollution_agent.py
    ├── parent_agent.py
```

---

# 📜 Code Files

## `config/settings.py`

```python
import os
from dotenv import load_dotenv

# Load env vars from .env
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "dummy-key")
DEFAULT_MODEL = "gpt-4o"

# Agent configurations
AGENT_CONFIG = {
    "weather": {
        "llm": "openai",
        "tools": ["get_weather_city", "get_weather_country"],
        "mcp_servers": ["server1", "server2"]
    },
    "pollution": {
        "llm": "gemini",
        "tools": ["get_city_pollution", "get_country_pollution"],
        "mcp_servers": ["server3", "server4"]
    }
}
```

---

## `tools/weather_tools.py`

```python
from mcp.server1 import fetch_weather

def get_weather_city(city: str) -> str:
    """Fetch weather for a city using MCP (server1)."""
    return fetch_weather(city)

def get_weather_country(country: str) -> str:
    """Mock weather for a country."""
    return f"Average weather in {country} is Mild 🌤️"
```

---

## `tools/pollution_tools.py`

```python
def get_city_pollution(city: str) -> str:
    return f"Pollution level in {city} is Moderate (AQI 120)"

def get_country_pollution(country: str) -> str:
    return f"Average AQI in {country} is 95"
```

---

## `mcp/server1.py`  ✅ Weather MCP client

```python
import requests

def fetch_weather(location: str) -> str:
    """
    Fetch weather from wttr.in for the given location.
    Example: https://wttr.in/London?format=3
    """
    try:
        url = f"https://wttr.in/{location}?format=3"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            return response.text.strip()
        return f"Weather service error: {response.status_code}"
    except Exception as e:
        return f"Weather fetch failed: {e}"
```

---

## `mcp/server2.py`

```python
def dummy_server2(task: str) -> str:
    return f"Server2 placeholder executed task: {task}"
```

---

## `mcp/server3.py`

```python
def dummy_server3(task: str) -> str:
    return f"Server3 placeholder executed task: {task}"
```

---

## `mcp/server4.py`

```python
def dummy_server4(task: str) -> str:
    return f"Server4 placeholder executed task: {task}"
```

---

## `agents/weather_agent.py`

```python
from tools.weather_tools import get_weather_city, get_weather_country

def create_weather_agent():
    def run(task: dict):
        if "city" in task:
            return get_weather_city(task["city"])
        if "country" in task:
            return get_weather_country(task["country"])
        return "Weather agent: No valid input found"
    return run
```

---

## `agents/pollution_agent.py`

```python
from tools.pollution_tools import get_city_pollution, get_country_pollution

def create_pollution_agent():
    def run(task: dict):
        if "city" in task:
            return get_city_pollution(task["city"])
        if "country" in task:
            return get_country_pollution(task["country"])
        return "Pollution agent: No valid input found"
    return run
```

---

## `agents/agent_factory.py`

```python
from agents.weather_agent import create_weather_agent
from agents.pollution_agent import create_pollution_agent

class AgentFactory:
    def __init__(self):
        self.registry = {
            "weather": create_weather_agent,
            "pollution": create_pollution_agent
        }

    def get_agent(self, name: str):
        if name not in self.registry:
            raise ValueError(f"Agent '{name}' not registered.")
        return self.registry[name]()
```

---

## `agents/parent_agent.py`

```python
from config.settings import AGENT_CONFIG
from agents.agent_factory import AgentFactory

def parse_prompt(prompt: str) -> dict:
    """Rule-based parser to extract task from user prompt."""
    prompt = prompt.lower()

    if "weather" in prompt:
        if "in" in prompt:
            city = prompt.split("in")[-1].strip()
            return {"topic": "weather", "city": city}
        return {"topic": "weather", "country": "India"}

    if "pollution" in prompt:
        if "in" in prompt:
            city = prompt.split("in")[-1].strip()
            return {"topic": "pollution", "city": city}
        return {"topic": "pollution", "country": "India"}

    return {"topic": "unknown"}

def create_parent_agent():
    factory = AgentFactory()

    def run(prompt: str):
        task = parse_prompt(prompt)
        topic = task["topic"]

        if topic not in AGENT_CONFIG:
            return f"Parent agent: Unknown topic from prompt: '{prompt}'"

        agent = factory.get_agent(topic)
        return agent(task)

    return run
```

---

## `main.py`

```python
from agents.parent_agent import create_parent_agent

def main():
    parent = create_parent_agent()

    prompts = [
        "What is the weather in London?",
        "Give me the average weather in India",
        "Check pollution in Mumbai",
        "Tell me the country pollution for India",
        "How is the traffic in Delhi?"
    ]

    for p in prompts:
        print(f"\n📝 Prompt: {p}")
        print("🤖 Response:", parent(p))

if __name__ == "__main__":
    main()
```

---

# ✅ How to Run

1. Install dependencies:

   ```bash
   pip install requests python-dotenv
   ```
2. Run:

   ```bash
   cd order_mgmt_agents
   python main.py
   ```
3. You’ll see **real weather data from wttr.in** for city queries 🎉

---

👉 Do you also want me to **integrate LangGraph** so the **parent agent → weather agent / pollution agent routing** can be visualized in Mermaid/PNG?
